# Lab 00 · From rules to LLM calls

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Save `DEEPSEEK_API_KEY` in Colab Secrets and keep `PROVIDER="deepseek"`. You can also select `"gemini"` or `"freeinference"` with the corresponding secret. `ask` stops after 100 calls; optional SDK and framework examples make separate calls.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Rules and an LLM

A rule chooses a drink based on the temperature. A large language model (LLM) generates a reply from the text you send it, called a prompt. Here the setup cell gives you an `ask()` function that sends a prompt to the selected model service and returns its text reply. Run this cell to see `Rule: water` at 30°C and read the model’s suggestion. Change `temperature` to 20 and rerun. The rule now picks tea. Does the model make the same choice?

In [ ]:
temperature = 30
if temperature >= 25:
    rule = "water"
else:
    rule = "tea"
print("Rule:", rule)

answer = ask(f"It is {temperature}°C outside. Suggest water or tea and briefly explain why.")
print("LLM:", answer)

**Next:** Next, compare Python rules and model replies on the same microservices quiz.

## One set of questions, two ways to answer
ShopEasy's services talk to each other over a network. One request might time out, a slow service might tie up resources, and the web and mobile apps might need different data.

First answer seven short questions about these situations using Python rules. Then ask a large language model (LLM) the same questions and count where the answers agree. In Step 5, see how examples and a role instruction change the prompt for one question.

**First time through:** Run the warm-up and Steps 1–5. The cells marked Optional show other ways to make the call from Step 3. Each core model call sends a prompt and returns text. Later labs build agents that can choose actions.

**At a glance**

![Lab 00 at a glance: 7 quiz questions (retries + service patterns), then Apply the rule (rule_answer()) or Ask a model (ask() for one letter), then Compare (agreement count), then Prompting variants (0 / 1 / 3 demos · persona), then Inspect replies (same question per comparison).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/00.svg)

## Think about the example

A retry repeats a failed request; a timeout can leave the result uncertain. Bulkheads separate resources, an API gateway handles shared edge concerns, and a BFF adapts an API to a client.

**Concepts:** Safe retries and idempotency; Bulkheads; API gateways and BFFs.

Use the scenario above to consider these questions before running the code:

1. When is a retry safe, and why is a timed-out payment uncertain?
2. How do bulkheads, API gateways, and BFFs address different problems?

## The idea in context

A Python rule checks explicit facts; an LLM reads a natural-language question and returns a response. Both answers need to be reviewed in context.

Retries need a transient failure, a safe operation, a limit, and backoff. Bulkheads isolate resources; gateways and BFFs serve different edge needs.

Zero-shot, one-shot, and few-shot prompts include zero, one, or several worked examples. Persona prompting adds a role or perspective; it is a separate choice from example count.

| Key term | Meaning |
| --- | --- |
| **Idempotency** | Repeating an operation has the same intended effect as performing it once. |
| **Bulkhead** | A resource boundary that keeps one dependency from exhausting another’s capacity. |
| **API gateway** | An entry point that can route external requests and apply shared policies. |
| **Backend for frontend (BFF)** | A backend that adapts an API to a particular client’s needs. |
| **LLM prompt** | The instructions, facts, and question sent to a language model. |
| **Zero-shot** | A prompt with no worked input/output examples. |
| **One-shot** | A prompt with one worked input/output example; also called single-shot. |
| **Few-shot** | A prompt with several worked input/output examples. |
| **Persona prompting** | Asking the model to adopt a role or perspective; this does not guarantee expertise. |

## Main exercise at a glance

1. Read the questions
2. Apply explicit rules
3. Make one LLM call
4. Ask an LLM the same questions
5. Compare zero-shot, one-shot, few-shot, and persona prompts

## Step 1 · Read the questions
Run the cell to print the seven questions and their answer choices. Pick your own answers before running Step 2. The fields `kind`, `transient`, and `idempotent` are clues the Python rule uses later.

A **transient failure** is temporary. An operation is **idempotent** if repeating it has the same intended effect as doing it once. A timed-out card charge may already have succeeded. The idempotency key is an identifier sent with the original request and reused on retries. In this exercise, the server recognizes it and prevents a second charge.

In [ ]:
retry_choices = ["A. Retry with a limit and backoff", "B. Do not retry automatically"]
pattern_choices = ["A. Bulkhead", "B. API gateway", "C. BFF"]
cases = [
    {"name": "Product read", "kind": "retry", "transient": True, "idempotent": True,
     "question": "A read-only product lookup times out due to a temporary network failure. Retry?", "choices": retry_choices},
    {"name": "Unsafe payment", "kind": "retry", "transient": True, "idempotent": False,
     "question": "A temporary network failure causes a card charge to time out. Repeating it may charge twice. Retry?", "choices": retry_choices},
    {"name": "Safe payment", "kind": "retry", "transient": True, "idempotent": True,
     "question": "A charge times out due to a temporary network failure. The server guarantees deduplication using the same idempotency key. Retry?", "choices": retry_choices},
    {"name": "Invalid request", "kind": "retry", "transient": False, "idempotent": True,
     "question": "A product read gets a permanent invalid-request error. Retry unchanged?", "choices": retry_choices},
    {"name": "Resource isolation", "kind": "isolate_resources",
     "question": "Slow recommendations exhaust the shared request pool. What isolates resources?", "choices": pattern_choices},
    {"name": "Client needs", "kind": "client_specific_api",
     "question": "Mobile needs small responses; web needs rich pages. What provides client-specific APIs?", "choices": pattern_choices},
    {"name": "Shared entry", "kind": "shared_entry",
     "question": "External clients need one entry point for routing and common authentication. What fits?", "choices": pattern_choices},
]
for case in cases:
    print(f"\n{case['name']}: {case['question']}")
    print("  " + " | ".join(case["choices"]))

## Step 2 · Apply explicit rules
Read the `if` statements and predict what they print. The retry rule picks A only when both facts are true: the failure is temporary and repeating the operation is safe. **Backoff** means waiting before retrying, usually with longer waits on later attempts. Real systems also limit attempts and often vary the wait slightly to keep clients from retrying together.

For the other questions, a **bulkhead** gives a dependency its own resource pool. An **API gateway** gives clients a shared entry point for routing and common checks. A **backend for frontend (BFF)** adapts an API to the needs of one client type. A system can use a gateway and a BFF together.

The output is one answer letter per case. Compare each letter with your own choice.

In [ ]:
rule_answers = {}
for case in cases:
    kind = case["kind"]
    if kind == "retry":
        if case["transient"] and case["idempotent"]:
            answer = "A"
        else:
            answer = "B"
    elif kind == "isolate_resources":
        answer = "A"
    elif kind == "client_specific_api":
        answer = "C"
    elif kind == "shared_entry":
        answer = "B"
    else:
        raise ValueError("Unknown case kind")
    rule_answers[case["name"]] = answer
    print(case["name"], "->", answer)

## Step 3 · Make one LLM call
Read the text assigned to `prompt`. It gives the model a situation, three answer choices, and instructions to reply with one letter. `ask(prompt)` sends one request and returns the reply as a Python string.

The classroom answer is **A** for Bulkhead. Check both the answer and whether the reply follows the one-letter instruction. You can go straight to Step 4 after this cell.

In [ ]:
prompt = """Slow recommendations exhaust the shared request pool.
Which pattern isolates resources so other services can keep working?
A. Bulkhead
B. API gateway
C. BFF
Reply with only the letter of one listed option."""

reply = ask(prompt)
print("Model reply:", reply)

## Step 3a · Optional: the same prompt with an SDK
A **software development kit (SDK)** is a library you can use to call a service from code. `ask()` handles those details for you. Here the OpenAI Python SDK shows the model name, the messages we send, and where the returned text lives. The request goes to your selected provider using its key and the shared request format.

Run Step 3 first so `prompt` exists. The cell installs the library if needed and sends one additional request. Look at the two messages. The system message gives the model a standing instruction, and the user message contains the quiz question.

In [ ]:
if find_spec("openai") is None:
    %pip install -q "openai==2.8.1"
from openai import OpenAI
from coursekit import PROVIDERS, secret

settings = PROVIDERS[PROVIDER]
options = settings.get("extra_body") or {}
if PROVIDER == "freeinference":
    options = {"thinking": {"type": "disabled"}}
client = OpenAI(api_key=secret(settings["secret_name"]),
                base_url=settings["base_url"], timeout=45.0, max_retries=0)
messages = [{"role": "system", "content": "You are a concise teaching assistant."},
            {"role": "user", "content": prompt}]
response = client.chat.completions.create(
    model=settings["model"], messages=messages, max_tokens=256, extra_body=options)
print("SDK reply:", response.choices[0].message.content)

## Step 3b · Optional: the same prompt with LangChain
LangChain is a framework that offers the same calling interface for different model services. Its `ChatOpenAI` adapter sends the messages, and `invoke(messages)` returns an object whose `content` holds the reply. Run Step 3 first to create the question, then compare this output with the helper or SDK output.

The SDK and LangChain examples each make a new request outside the helper's call budget. Their replies may differ and the provider may charge for the requests.

In [ ]:
if find_spec("langchain_openai") is None:
    %pip install -q "langchain-openai==1.1.0" "openai==2.8.1"
from langchain_openai import ChatOpenAI
from coursekit import PROVIDERS, secret

settings = PROVIDERS[PROVIDER]
options = settings.get("extra_body") or {}
if PROVIDER == "freeinference":
    options = {"thinking": {"type": "disabled"}}
model = ChatOpenAI(model=settings["model"], base_url=settings["base_url"],
                   api_key=secret(settings["secret_name"]), extra_body=options,
                   max_tokens=256, timeout=45, max_retries=0)
messages = [("system", "You are a concise teaching assistant."), ("user", prompt)]
response = model.invoke(messages)
print("LangChain reply:", response.content)

## Step 3c · Optional: DeepSeek Harness SDK
**Why a harness?** A basic SDK sends a model request. An agent harness can manage a session and connect the model to tools. This example runs the same question through the [DeepSeek Harness Python SDK](https://github.com/deepseek-ai/deepseek-harness/blob/master/docs/user/guide/python-sdk.md). The harness supplies its own system instructions and session management, so this is not an identical request to Step 3a.

Run Step 3 first so `prompt` exists. Select `PROVIDER="deepseek"`, provide `DEEPSEEK_API_KEY` in Colab Secrets, and change `RUN_HARNESS` below to `True`. The optional example remains off until you choose to run it. The first run installs the pinned preview SDK and its runtime (about 90 MB). If a different version is already installed, install the pinned version and restart the runtime before continuing. It may make additional model requests and use more tokens than a single API call.

For this introductory question the code disables the Harness shell, file, and working-directory tools and the optional session-log and package-inventory uploads. It uses a fresh temporary workspace and removes the session files afterward; a temporary folder alone is not a sandbox. A real tool-using agent needs explicitly chosen permissions and verification. This example does not execute a tool.

In [ ]:
RUN_HARNESS = False  # Set True to try the live Harness example.
if not RUN_HARNESS or PROVIDER != "deepseek":
    print('Set RUN_HARNESS = True and PROVIDER = "deepseek" to run this example.')
else:
    if find_spec("deepseek_harness") is None:
        %pip install -q "deepseek-harness-sdk==0.2.2rc2"
    from pathlib import Path
    from importlib.metadata import version
    assert version("deepseek-harness-sdk") == version("deepseek-harness-runtime-bin") == "0.2.2rc2", "Install the pinned SDK and restart the runtime."
    from tempfile import TemporaryDirectory
    from deepseek_harness import DeepSeekHarness
    from coursekit import secret
    with TemporaryDirectory(prefix="lab00-harness-") as workspace:
        patch = Path(workspace) / "no-tools.yml"
        disabled = ("persistent-bash", "persistent-pwsh", "tool-fs", "tool-working-directory", "session-log-deepseek", "plugin-package-inventory-deepseek")
        patch.write_text("".join(f"- id: {name}\n  disabled: true\n" for name in disabled))
        with DeepSeekHarness(provider="deepseek-official", model="deepseek-flash",
                             profile="sdk-minimal", cwd=workspace, dsh_home=str(Path(workspace) / "home"),
                             patches=(str(patch),), api_key=secret("DEEPSEEK_API_KEY"),
                             max_tokens=4096, reasoning_effort="low") as harness:
            result = harness.run(prompt, session_id="bulkhead-question")
            assert result.finish_reason in (None, "completed"), "Harness did not complete."
            print("Harness reply:", result.final_response)

## Step 4 · Ask an LLM the same questions
The loop sends each question and its choices to the model in a separate request. Run Steps 1 and 2 first so Python has the questions and rule answers.

Look for `rule=`, `model=`, and `recognized=` on each line. The first total counts replies that contain only an offered letter. The second counts replies that match the rule’s choice. Python can recognize a letter, an exact option text, or the complete labeled option. It leaves mixed or unrecognized replies unresolved for you to inspect.

In [ ]:
format_matches, agreement = 0, 0
for case in cases:
    choice_lines = "\n".join(case["choices"])
    question_prompt = f"{case['question']}\n{choice_lines}\nReply with only the letter of one listed option."
    reply = ask(question_prompt).strip()
    normalized = " ".join(reply.upper().split()).rstrip(".")
    recognized = None
    for choice in case["choices"]:
        letter, text = choice.split(". ", 1)
        if reply == letter:
            format_matches += 1
        if normalized in (letter, text.upper(), choice.upper()):
            recognized = letter
    expected = rule_answers[case["name"]]
    if recognized == expected:
        agreement += 1
    print(f"{case['name']}: rule={expected}, model={reply}, recognized={recognized}")
print(f"Replies in the requested format: {format_matches}/{len(cases)}")
print(f"Recognizable decisions matching the rules: {agreement}/{len(cases)}")

## Step 5 · Compare zero-shot, one-shot, few-shot, and persona prompts
A **shot** is a worked input → output example included in a prompt. In this lab a worked example includes both a question and its answer. The model sees those examples as part of the current request.

| Technique | What the prompt contains |
| --- | --- |
| **Zero-shot** (no-shot) | Instructions and the new question, with no worked examples. |
| **One-shot** (single-shot) | The same instructions and new question, plus one worked question → answer example. |
| **Few-shot** | The same instructions and new question, plus several worked examples: three here. |
| **Persona / role prompting** | An added role or perspective, such as a mobile developer. A persona can be combined with any example count. |

![Two prompt comparisons: zero, one, or three worked examples for a support ticket, followed by neutral and mobile developer prompts for a shared architecture question.](https://mkassaf.github.io/agentic-ai-course/assets/images/prompt-techniques.svg)

The first comparison makes **three calls** about the same late package ticket. ShopEasy’s support desk uses made-up queue names. With zero examples the names alone give the model little basis for a decision. One example shows a billing ticket. Three examples also show a delivery ticket and an account ticket. The expected label is **MOSS**, the queue for delivery issues. Compare what each prompt gives the model enough information to infer. A model could guess the right label with any variant.

The second comparison makes **two more calls** on a different task. Both prompts ask for the same architecture choice and one operational trade-off. Only the second adds “You are a mobile developer.” Compare which effects the replies emphasize. “Neutral” here means the helper’s regular system instruction with no additional role in the user prompt. Lab 01 shows how to place a role in a separate system message.

Run setup first. This cell sends five additional requests through `ask()`.

In [ ]:
technique_question = "My package was supposed to arrive yesterday, but it has not arrived."
label_request = "Route the support ticket. Reply with one queue label: KITE, MOSS, or LAMP."
demonstrations = [
    "Q: I was charged twice for one order.\nA: KITE",
    "Q: The courier left my parcel at the wrong address.\nA: MOSS",
    "Q: I cannot sign in after resetting my password.\nA: LAMP",
]
shot_prompts = {
    "Zero-shot": "",
    "One-shot": demonstrations[0],
    "Few-shot": "\n\n".join(demonstrations),
}
for name, examples in shot_prompts.items():
    technique_prompt = f"{label_request}\n{examples}\nQ: {technique_question}\nA:"
    technique_reply = ask(technique_prompt).strip()
    print(name, ":", technique_reply, "| classroom answer: MOSS")

architecture_question = "ShopEasy has phone and browser clients that need different tailored APIs. Which pattern fits?"
persona_task = (f"{architecture_question} Choose Bulkhead, API gateway, or BFF. "
                "Explain one operational trade-off in two short sentences.")
for name, role in [("Neutral", ""), ("Persona", "You are a mobile developer.")]:
    print(name, ":", ask(f"{role}\n{persona_task}"))

## What you should see
In Step 4, the rule answers are **A, B, A, B, A, C, B**. The resource isolation question expects `A`. A reply of `A. Bulkhead` matches the decision and misses the requested format. A conflicting reply such as `A. BFF` stays unresolved.

Step 5 prints three queue guesses and two architecture replies. Compare the guesses with `MOSS`, then read whether the mobile developer role changes the trade-off the model emphasizes. These five replies give you examples to inspect.

## Try it
1. Explain why a timed-out card charge without a server-supported idempotency guarantee must not be retried blindly. How could you check its status?
2. Change the product lookup's `transient` fact to `False` **and update its question** to describe a permanent error. Rerun all steps. Why does the rule change?
3. Add a new client-specific API scenario. Predict the rule’s answer, then compare it with the model.
4. Compare the helper, SDK, LangChain and Harness replies. Which lines send the prompt in each example? Does a matching letter guarantee the design is correct?
5. In Step 5, use the delivery example as the only demonstration and predict what will change. Then try a different persona and compare the architecture trade-offs.

## Takeaway
Rules are transparent when the relevant facts are explicit; an LLM can interpret a question written in natural language, but its answer still needs checking. Shots vary the number of worked examples; a persona varies the requested perspective. Both still need checking. Next, Lab 01 separates system and user prompts, carries context forward, and validates an answer.